# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a Croissant-standard dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset basic metadata
metadata = dataset.metadata  # This is an mlcroissant.DatasetMetadata object
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Cite as: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets (tables-like structures) with their @ids and fields

print("Available record sets and their fields (by @id):\n")
record_sets = dataset.metadata.record_sets

record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}\n  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} [@id: {field.id}] (column: {field.column.id if field.column is not None else 'N/A'})")
    print()

# For illustration, print the records from the first record set
if record_sets:
    print(f"Sample records from the first RecordSet (@id={record_sets[0].id}):\n")
    for i, record in enumerate(dataset.records(record_set=record_sets[0].id)):
        print(record)
        if i >= 2:  # Show just 3 records
            break

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame. Use the `@id`s collected above for referencing record sets and fields.

In [ ]:
# Extract records from all record sets into DataFrames

dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"RecordSet '@id': {rs.id}, name: {rs.name}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3))
    print()

# For further steps, pick the main protein abundance record set
# Let's try to pick the first one if unsure
main_record_set_id = record_sets[0].id if record_sets else None
print(f"Main analysis will use RecordSet '@id': {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter, normalize, and group by key attributes.

In [ ]:
import numpy as np

# Choose numeric fields from the main record set for analysis (by @id)
main_df = dataframes[main_record_set_id]
print("Available numeric-like fields in main record set:")
numeric_candidates = [col for col in main_df.columns if np.issubdtype(main_df[col].dtype, np.number)]
print(numeric_candidates)

# Example: Use 'coverage_percent' or fallback to any numeric field
numeric_field = numeric_candidates[0] if numeric_candidates else None
if numeric_field is None:
    print("No numeric field found for filtering.")
else:
    print(f"\nPerforming EDA using field: {numeric_field}\n")
    # Filter records where numeric_field > threshold
    threshold = main_df[numeric_field].mean()
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()

    print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean):")
    print(filtered_df[[numeric_field]].head())

    # Normalize the field (mean=0, std=1)
    normalized_name = f"{numeric_field}_normalized"
    filtered_df[normalized_name] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized field {numeric_field}:")
    print(filtered_df[[numeric_field, normalized_name]].head())

    # Attempt to group by a categorical field if present (e.g., 'accession', 'protein_id', etc.)
    group_candidates = [col for col in main_df.columns if col != numeric_field and main_df[col].dtype == object]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize the numeric distribution and categorical grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of `{numeric_field}`')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if available
    if group_field:
        # Show only top 10 groups for readability
        group_counts = filtered_df[group_field].value_counts().head(10).index
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df[filtered_df[group_field].isin(group_counts)], x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field} (top 10 groups)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR^2 protein dataset using `mlcroissant`, explored its schema and records using `@id` references, extracted data for analysis, performed basic EDA, and visualized field distributions. For full details on fields, columns, and advanced analytics, see the [FAIR^2 Croissant schema documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and `mlcroissant` [documentation](https://mlcommons.github.io/croissant/).

**Key Findings:**
- The dataset enables quantitative analysis of mass spectrometry protein abundance results, with feature-rich metadata and processing provenance.
- Filtering and normalization of numeric fields enable rapid profiling and quality control.

For further exploration, target specific record sets and columns by their `@id`, and leverage `mlcroissant` for scalable FAIR data engineering!